In [2]:
import pandas as pd 
import folium

#Cargando informacion
data = pd.read_csv("denue_inegi_62_.csv", encoding="latin-1")

Servicios clasificados para atencion inmediata:

- 621111,Consultorios de medicina general del sector privado
- 621112,Consultorios de medicina general del sector público
- 621115,Clínicas de consultorios médicos del sector privado
- 621116,Clínicas de consultorios médicos del sector público
- 621491,Otros centros del sector privado para la atención de pacientes que no requieren hospitalización
- 621492,Otros centros del sector público para la atención de pacientes que no requieren hospitalización
- 622111,Hospitales generales del sector privado
- 622112,Hospitales generales del sector públicoss

In [3]:
#Usando data de ciudad de mexico
filter_data = data[data["cve_ent"].isin([9])]

#Usando solo los servicios de atencion primaria/inmediata
codigos = [
    621111, 621112, 621115, 621116, 
    621491, 621492, 622111, 622112
]
df = filter_data[filter_data['codigo_act'].isin(codigos)]

In [4]:
df.shape

(3851, 42)

In [4]:
unique_mapping = df[['codigo_act', 'nombre_act']].drop_duplicates().sort_values(by='codigo_act')

unique_mapping.to_csv('unique_codigo_act_nombre_act.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Map with points

cdmx_coords = [19.4326, -99.1332]
codigos_privados = [621111, 621115, 621491, 622111]

mapa_cdmx = folium.Map(location=cdmx_coords, zoom_start=11, tiles='OpenStreetMap')

for index, row in df.iterrows():
    lat = row['latitud']
    lon = row['longitud']
    codigo = row['codigo_act']

    if pd.notna(lat) and pd.notna(lon):
        if codigo in codigos_privados:
            color_punto = 'blue'  # Privado
        else:
            color_punto = 'red'
        folium.CircleMarker(
            location=[lat, lon],
            radius=5, # Tamaño del punto
            color=color_punto, # Color del borde del punto
            fill=True,
            fill_color=color_punto, # Color de relleno
            fill_opacity=0.6,
            #popup=f"ID: {row['id']}"
            popup=f"Location {row['longitud'], row['latitud']}" # Texto que aparece al dar clic en el punto
        ).add_to(mapa_cdmx)
        
nombre_archivo = 'mapa_lugares_cdmx.html'
mapa_cdmx.save(nombre_archivo)

In [36]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt

#Plot a pipe map from a starting point and and time

def generar_mapa_interactivo(lat_centro, lon_centro, radio_metros_descarga, radio_tiempo_tda):
    print("1. Descargando el grafo dirigido real desde OpenStreetMap...")
    G_dirigido = ox.graph_from_point((lat_centro, lon_centro), dist=radio_metros_descarga, network_type='drive')
    
    G_dirigido = ox.add_edge_speeds(G_dirigido)
    G_dirigido = ox.add_edge_travel_times(G_dirigido)

    print("2. Construyendo el Espacio Geométrico Simétrico (TDA)...")
    G_simetrico = nx.Graph()
    
    # Heredar metadatos espaciales (Coordenadas y CRS)
    G_simetrico.graph = G_dirigido.graph.copy() 
    G_simetrico.add_nodes_from(G_dirigido.nodes(data=True))
    
    for u, v, data in G_dirigido.edges(data=True):
        tiempo = data.get('travel_time', 10) 
        
        if G_simetrico.has_edge(u, v):
            peso_actual = G_simetrico[u][v]['peso_tda']
            G_simetrico[u][v]['peso_tda'] = (peso_actual + tiempo) / 2
        else:
            G_simetrico.add_edge(u, v, weight=tiempo, peso_tda=tiempo, geometry=data.get('geometry'))

    print("3. Encontrando el nodo central...")
    nodo_origen = ox.distance.nearest_nodes(G_dirigido, X=lon_centro, Y=lat_centro)

    print("4. Ejecutando BFS Topológico...")
    subgrafo_cluster = nx.ego_graph(
        G_simetrico, 
        nodo_origen, 
        radius=radio_tiempo_tda, 
        distance='peso_tda'
    )
    print(f" -> {len(subgrafo_cluster.nodes)} intersecciones alcanzables encontradas.")

    print("5. Construyendo el mapa interactivo con Folium...")
    
    # Origin node
    lat_origen_real = G_dirigido.nodes[nodo_origen]['y']
    lon_origen_real = G_dirigido.nodes[nodo_origen]['x']

    # Inicializamos el mapa centrado en el origen
    mapa = folium.Map(location=[lat_origen_real, lon_origen_real], zoom_start=14, tiles='OpenStreetMap')

    # Convertimos el subgrafo a MultiDiGraph para poder extraer sus geometrías
    subgrafo_render = nx.MultiDiGraph(subgrafo_cluster)
    subgrafo_render.graph = subgrafo_cluster.graph.copy()
    
    # Extraemos las calles (aristas) como un GeoDataFrame
    _, gdf_edges = ox.graph_to_gdfs(subgrafo_render)

    # Añadimos la red de calles alcanzables al mapa
    folium.GeoJson(
        gdf_edges,
        style_function=lambda feature: {
            'color': "#5C2900", # Cyan brillante
            'weight': 3,
            'opacity': 0.6
        },
        name="Clúster Topológico (Calles)"
    ).add_to(mapa)

    # Añadimos el Punto de Origen con un marcador distinto
    folium.CircleMarker(
        location=[lat_origen_real, lon_origen_real],
        radius=7,                 # Tamaño del punto
        color='red',              # Borde rojo
        weight=2,
        fill=True,
        fill_color='#FF0000',     # Relleno rojo sólido
        fill_opacity=1.0,
        tooltip="Punto de Origen (Centro del Clúster)",
        popup=f"Lat: {lat_origen_real:.4f}<br>Lon: {lon_origen_real:.4f}",
        name="Origen"
    ).add_to(mapa)

    folium.LayerControl().add_to(mapa)

    # Guardar en archivo
    nombre_archivo = "pipe_map.html"
    mapa.save(nombre_archivo)

In [37]:
lat = 19.28575742
long = -99.13255436

km_around = 3000

time = 700

generar_mapa_interactivo(lat, long, km_around, time)

1. Descargando el grafo dirigido real desde OpenStreetMap...
2. Construyendo el Espacio Geométrico Simétrico (TDA)...
3. Encontrando el nodo central...
4. Ejecutando BFS Topológico...
 -> 5467 intersecciones alcanzables encontradas.
5. Construyendo el mapa interactivo con Folium...
